# Phase 1 - Frozen ResNet50 baseline on Stanford Online Products

Embeds the 60,502 test images with a pretrained ResNet50 (no training), indexes them
with FAISS, and reports Recall@1/5/10 and mAP@100. This is the reference number every
trained model must beat.

**Kaggle setup before running:**
1. *Add Data* -> search `stanford online products` -> add the dataset
2. *Settings* -> Accelerator -> **GPU T4 x2** (or P100)
3. Make the `src/vpse` package reachable - either set `REPO_URL` below to your GitHub
   repo, or upload the repo as a Kaggle dataset and it will be auto-detected.

In [ ]:
!pip install -q faiss-cpu

In [ ]:
import glob, os, subprocess, sys
from pathlib import Path

REPO_URL = ''  # e.g. 'https://github.com/Akash-5675/visual-product-search.git'

# --- locate the SOP dataset (folder holding Ebay_train.txt) ---
hits = glob.glob('/kaggle/input/**/Ebay_train.txt', recursive=True) or \
       glob.glob('../data/**/Ebay_train.txt', recursive=True)
assert hits, 'SOP dataset not found - use Add Data to attach it'
DATA_ROOT = Path(hits[0]).parent

# --- locate the vpse package ---
if REPO_URL:
    if not Path('/kaggle/working/repo').exists():
        subprocess.run(['git', 'clone', '-q', REPO_URL, '/kaggle/working/repo'], check=True)
    SRC = Path('/kaggle/working/repo/src')
else:
    found = glob.glob('/kaggle/input/**/vpse/config.py', recursive=True) or \
            glob.glob('../src/vpse/config.py', recursive=True)
    assert found, 'vpse package not found - set REPO_URL or upload the repo as a dataset'
    SRC = Path(found[0]).parents[1]

sys.path.insert(0, str(SRC))
print('data:', DATA_ROOT)
print('src :', SRC)

In [ ]:
import torch
from vpse.data.sop import SOPDataset, eval_transform, load_split

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

# Phase 0 sanity check: the splits must not share any product
tr, te = load_split(DATA_ROOT, 'train'), load_split(DATA_ROOT, 'test')
print(f'{len(tr)} train / {len(te)} test images')
print(f'{tr.class_id.nunique()} train / {te.class_id.nunique()} test products')
print('classes disjoint:', set(tr.class_id).isdisjoint(set(te.class_id)))

In [ ]:
import json

from vpse.models.embedder import Embedder
from vpse.retrieval.eval import evaluate, leave_one_out_neighbors
from vpse.retrieval.index import embed_dataset, save_embeddings

test_ds = SOPDataset(DATA_ROOT, 'test', eval_transform())

# use_head=False -> raw 2048-d pool5 features, genuinely untrained
model = Embedder(freeze_backbone=True, use_head=False)
embs, labels = embed_dataset(model, test_ds, device, batch_size=256, num_workers=4)

save_embeddings(Path('/kaggle/working/cache/baseline_test.npz'), embs, labels)
metrics = evaluate(embs, labels)
print(json.dumps(metrics, indent=2))
Path('/kaggle/working/baseline.json').write_text(json.dumps(metrics, indent=2))

In [ ]:
%matplotlib inline
neighbors = leave_one_out_neighbors(embs, k_max=5)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

rng = np.random.default_rng(0)
queries = rng.choice(len(test_ds), 6, replace=False)
fig, axes = plt.subplots(6, 6, figsize=(13, 14))
for r, q in enumerate(queries):
    for c, idx in enumerate([q] + neighbors[q, :5].tolist()):
        ax = axes[r, c]
        ax.imshow(Image.open(test_ds.data_root / test_ds.df.iloc[idx]['path']).convert('RGB'))
        ax.axis('off')
        if c == 0:
            ax.set_title('query', fontsize=9)
        else:
            hit = labels[idx] == labels[q]
            ax.set_title('match' if hit else 'wrong', fontsize=9,
                         color='green' if hit else 'red')
fig.tight_layout()
fig.savefig('/kaggle/working/baseline_grid.png', dpi=120)
plt.show()

## Phase 2 - metric learning

Run each loss in turn and record Recall@k for the results table. Batch size drives
mining quality, so use the largest `batch_p * batch_k` that fits in GPU memory.
Each run saves the best checkpoint and its metrics to `/kaggle/working/results/`.

In [ ]:
from vpse.config import Config
from vpse.train import main as train_main

cfg = Config(
    data_root=DATA_ROOT,
    results_dir=Path('/kaggle/working/results'),
    loss='triplet_random',   # then 'triplet_hard', then 'arcface'
    epochs=15,
    batch_p=32, batch_k=4,   # 128 images per batch
    num_workers=4,
)
train_main(cfg)